# Model Preparation : XGBoost

Veuillez tester la préparation du modèle sur chaque fichier CSV nettoyé en amont et comparer les résultats obtenus.
- adapté au dataset de films suivant : [IMDB Dataset of Top 1000 Movies and TV Shows](https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows)

### Librairies requises

In [51]:
# !pip install scikit-learn
# !pip install pandas
!pip install xgboost


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
import pandas as pd
import joblib, os
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

## Préparation du modèle

In [53]:
xgboost_model = XGBClassifier(
    random_state=42
)

## Préparation de données pour entraînement

In [54]:
path = input("Entrer le chemin du dataset CSV pour l'entrainement: ")

if not path.endswith('.csv') or not path:
    path = "../data/top_movies_cleaned.csv"

In [55]:
def prepare_data(file_path, targeted_feature="GenreEncoded", test_size=0.2, random_state=42):
    dataframe = pd.read_csv(file_path)
    X = dataframe.drop(targeted_feature, axis=1)
    y = dataframe[targeted_feature]
    return train_test_split(X, y, test_size=0.2, random_state=42)

In [56]:
X_train, X_test, y_train, y_test = prepare_data(path)

## Entrainement du modèle

In [57]:
xgboost_model.fit(X_train, y_train)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Test du modèle

- construire un dictionnaire de mapping entre les genres encodés et les noms de genre correspondant.

In [58]:
def build_genre_mapping(dataframe, col_genre_name, list_classes):
    try:
        if col_genre_name in dataframe.columns and "GenreEncoded" in dataframe.columns:
            genre_map = (
                dataframe[["GenreEncoded", col_genre_name]]
                .drop_duplicates()
                .sort_values("GenreEncoded")
                .set_index("GenreEncoded")[col_genre_name]
                .to_dict()
            )
        else:
            # Fallback si la colonne n'existe pas
            genre_map = {int(c): f"genre_{int(c)}" for c in list_classes}
    except Exception as e:
        raise ValueError(f"Erreur lors de la construction du mapping: {str(e)}")
    
    return genre_map

In [59]:
y_pred = xgboost_model.predict(X_test)
y_proba = xgboost_model.predict_proba(X_test)
confidence_pct = y_proba.max(axis=1) * 100

# Construction du mapping encodage -> genre
df_full = pd.read_csv(path)

# Détecte la colonne de genre disponible
col_genre_name = "Genre"

# Utilise la fonction pour créer le mapping
genre_map = build_genre_mapping(df_full, col_genre_name, xgboost_model.classes_)

# Crée le DataFrame des prédictions
predictions_df = pd.DataFrame({
    "GenreEncoded_pred": y_pred,
    "genre_pred": pd.Series(y_pred).map(genre_map).values,
    "%_prediction": confidence_pct.round(2),
    "GenreEncoded_true": y_test.values,
    "genre_true": pd.Series(y_test.values).map(genre_map).values,
    "result": pd.Series(y_pred == y_test.values).map(
        {
            True: "Correct",
            False: "Incorrect"
        }).values
})

predictions_df.head(20)

,GenreEncoded_pred,genre_pred,%_prediction,GenreEncoded_true,genre_true,result
0,3,genre_3,55.080002,3,genre_3,Correct
1,3,genre_3,97.980003,3,genre_3,Correct
2,3,genre_3,67.550003,2,genre_2,Incorrect
3,1,genre_1,76.330002,3,genre_3,Incorrect
4,0,genre_0,99.809998,3,genre_3,Incorrect
5,1,genre_1,36.790001,0,genre_0,Incorrect
6,1,genre_1,47.380001,2,genre_2,Incorrect
7,0,genre_0,47.810001,3,genre_3,Incorrect
8,1,genre_1,98.019997,0,genre_0,Incorrect
9,2,genre_2,75.349998,3,genre_3,Incorrect


- score() → accuracy (précision) du modèle
    - exemple : si le modèle prédit correctement le genre de 80 films sur 100, alors l'accuracy serait de 80%.

In [60]:
score_dt = xgboost_model.score(X_test, y_test)
score_dt = round(score_dt * 100, 2)
print(f"Accuracy: {score_dt}%")

Accuracy: 50.0%


- f1-score → mesure de la précision du modèle en prenant en compte à la fois la précision et le rappel
    - exemple : si le modèle a une précision de 80% et un rappel de 70%, alors le f1-score serait d'environ 74%.

In [61]:
F1_score_dt = classification_report(y_test, y_pred, output_dict=True)
F1_score_dt = F1_score_dt['weighted avg']['f1-score']
print(f"F1-score: {F1_score_dt:.2f}")

F1-score: 0.51


- recall → mesure de la capacité du modèle à identifier correctement les classes positives
    - exemple : si le modèle prédit 100 films comme étant classés "Comedy" et que 80 d'entre eux sont effectivement classés "Comedy", 
        + alors le recall serait de 80%.

In [62]:
recall_dt = classification_report(y_test, y_pred, output_dict=True)
recall_dt = recall_dt['weighted avg']['recall']
print(f"Recall: {recall_dt:.2f}")

Recall: 0.50


- matrice de confusion → table pour comparer les prédictions du modèle avec les vraies étiquettes

In [63]:
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion :\n", cm)

Matrice de confusion :
 [[18  5  2  5]
 [ 3  7  1  7]
 [ 1  3  6  4]
 [ 8  9  5 22]]


- Rapport métriques par classe → précision, rappel et F1-score pour chaque classe

In [64]:
report = classification_report(y_test, y_pred)
report

'              precision    recall  f1-score   support\n\n           0       0.60      0.60      0.60        30\n           1       0.29      0.39      0.33        18\n           2       0.43      0.43      0.43        14\n           3       0.58      0.50      0.54        44\n\n    accuracy                           0.50       106\n   macro avg       0.47      0.48      0.47       106\nweighted avg       0.52      0.50      0.51       106\n'

## Interprétation des résultats

## Exportation du modèle

In [65]:
PATH_OUTPUT_MODEL = "../model"
if not os.path.exists(PATH_OUTPUT_MODEL):
    os.makedirs(PATH_OUTPUT_MODEL, exist_ok=True)

joblib.dump(xgboost_model, f"{PATH_OUTPUT_MODEL}/xgboost_model.pkl")

['../model/xgboost_model.pkl']

# Ressources

- [Decision Tree - scikit-learn](https://scikit-learn.org/stable/modules/tree.html)